# D02 · 客人从哪来

"从哪来"有三个层次，本册按由外到内的顺序拆：

1. **地理**：客源国。176 个国家，集中在少数几个。
2. **渠道**：客人通过谁订的（OTA、直订、线下旅行社、企业协议）。
3. **客群**：散客、协议客户、团队。

三层合起来决定了一件后面很关键的事：**取消率的基线**。
同样一家酒店，客源结构不同，取消率可以差十来个点，而这不是管理好坏能解释的。

先纠正一个口径问题：这份数据**没有酒店所在城市/地区字段**（只有客源国），
所以"地区"在这个项目里只能答成"客源国"，不能答成"酒店开在哪个城市"。

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hotel_insight import config, origin, viz
from hotel_insight.viz import FAINT, INK, MAIN, MAIN_LIGHT, MAIN_MID, MUTED

viz.apply_theme()
pd.set_option("display.width", 220)
pd.set_option("display.unicode.east_asian_width", True)

con = sqlite3.connect(config.DB_PATH)
s = config.load_settings()
a = s["analysis"]
NB = "D02"
SOURCE = "数据源：hotel.db / dwd_bookings（85,585 条入住记录，2015-07 至 2017-08）"

In [ ]:
# ---- 自检 ----
from hotel_insight.viz.theme import FONTS

d = pd.read_sql("SELECT * FROM dwd_bookings", con)
print(f"记录数 {len(d):,}；客源国 {d['country'].nunique()} 个；取消率 {d['is_canceled'].mean():.2%}")
print(f"中文字体 {'、'.join(FONTS)}")
assert len(d) == 85585 and any("CJK" in f for f in FONTS)

## 1 客源高度集中：10 个国家占八成

一个国家一个国家的条形图会画成 176 行，读不出重点。用帕累托图同时看"规模"和"累计覆盖"：
柱子是记录数，折线是累计占比，80% 那条参考线直接告诉你"要几个国家才够"。

In [ ]:
# ── 出图 D02_01：客源国帕累托 ──
# 为什么用帕累托而不是饼图：176 个国家画饼图会有 160 个看不清的扇区，
# 而且饼图无法回答"前几个够用"这个真问题；帕累托的累计线就是答案。
# 规范落在哪几行：双轴只保留 0/50/80/100 四个刻度 / 80% 参考线用灰不用红 / 末位是"其他"合计
# ---- 1 指标 ----
ct = origin.country_table(d, top_n=a["top_country_n"], min_n=a["min_group_n"])
head = ct.iloc[:-1]                       # 去掉"其他"：累计线只算真实国家
n80 = int((head["累计占比"] < 0.80).sum()) + 1          # 第几个国家时越过 80%
title = f"前 {a['top_country_n']} 国占 {head['占比'].sum():.0%}，第 {n80} 国时越过 80%"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"共 {d['country'].nunique()} 个客源国；末位'其他'是剩余 "
            f"{int(ct['记录数'].iloc[-1]):,} 条合计，占比 {ct['占比'].iloc[-1]:.0%}")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.8, 4.4))
viz.pareto(ax, head.index, head["记录数"].to_numpy(), cum=head["累计占比"].to_numpy())
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D02_01_客源国帕累托.png")
# ---- 4 读回断言 ----
bars = ax.patches
assert np.allclose([b.get_height() for b in bars], head["记录数"].to_numpy()), "柱高与记录数不一致"
assert len(ax.get_xticks()) == len(head) == a["top_country_n"]
print(f"第 {n80} 国累计 {head['累计占比'].iloc[n80 - 1]:.1%}，第 {n80 - 1} 国 {head['累计占比'].iloc[n80 - 2]:.1%}")
plt.show()

In [ ]:
# 国家明细表：规模、占比、取消率、房价、停留、提前期放一起
# 不用 DataFrame.style：环境里没装 jinja2，Styler 会直接报错（本项目不为此加依赖）
show = ct.copy()
for c in ["占比", "累计占比", "取消率", "下限", "上限"]:
    show[c] = (show[c] * 100).round(1)
show = show.rename(columns={"占比": "占比%", "累计占比": "累计%", "取消率": "取消率%",
                            "下限": "区间下%", "上限": "区间上%"})
show.round({"平均房价": 0, "平均停留": 1, "平均提前期": 0})

葡萄牙一地占三成，前四国过半，前十国八成。这意味着：

- 做营销，"全国市场"这种说法对这家酒店没有意义，实际就是三四个国家；
- 做取消分析，个别小国的极端值（样本 30 条算出 67%）必须抑制，否则会盖住主线；
- 做同比，只要葡萄牙的构成变了，整体指标就会晃，跟服务质量没关系。

最后一行的"其他"不是"某个国家"，是 166 个国家的合计。它的取消率必须用
**合并后的单数 ÷ 合并后的行数**重算，不能对各国比率取平均——后者会被小国放大。

## 2 客源国与酒店类型：两类客源各去各的店

同一个国家的人，是不是都去同一家店？把"国家 × 酒店类型"画成 100% 堆叠条，
一眼能看出哪些国家偏度假、哪些偏城市。

In [ ]:
# ── 出图 D02_02：客源国 × 酒店类型 ──
# 为什么用 100% 堆叠条而不是并排柱：问题是"这个国家的客人都去哪种店"，
# 分母应该是这个国家的总人数，并排柱会让读者在两个不同分母之间来回换算。
# 规范落在哪几行：行内占比 / 只在前 8 国上画（其余样本不足）/ 图例放底部不压数据
# ---- 1 指标 ----
top8 = head.index[:8].tolist()
mix = origin.mix_table(d[d["country"].isin(top8)], "country", "hotel_type")
mix = mix.loc[top8]                        # 按记录数从大到小，跟帕累托顺序一致
resort_hi = mix["度假酒店"].idxmax()
resort_lo = mix["度假酒店"].idxmin()
title = f"{resort_hi} {mix.loc[resort_hi, '度假酒店']:.0f}% 住度假店，{resort_lo}只有 {mix.loc[resort_lo, '度假酒店']:.0f}%"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = "口径：行内占比（%），仅前 8 大客源国；每行合计 100%"
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.6, 4.4))
viz.share_bar(ax, mix[["城市酒店", "度假酒店"]],
              colors=[viz.HOTEL_COLOR["城市酒店"], viz.HOTEL_COLOR["度假酒店"]])
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D02_02_客源国与酒店类型.png")
# ---- 4 读回断言：每行的两段加起来必须是 100 ----
tot = np.array([sum(ax.containers[k][i].get_width() for k in range(2)) for i in range(len(mix))])
assert np.allclose(tot, 100), f"堆叠条总长 {tot}，应为 100"
assert f"{mix.loc[resort_hi, '度假酒店']:.0f}%" in fig.texts[0].get_text()
plt.show()

两类客源很清楚：

- **偏度假店**：爱尔兰（65%）、英国（57%）、葡萄牙（53%）；
- **偏城市店**：意大利（86%）、法国（84%）、德国（83%）、比利时（81%）。

英国客人去度假店的比例高，但英国的取消率是全场最低（19.1%），
说明"度假客爱取消"这种直觉在这里不成立——真正的差别要看 D05 的渠道与提前期。

## 3 客源国的取消率差得比想象中大

同样的酒店、同样的年份，不同客源国的取消率从 18% 到 37%。
这个差距如果被当成"某国客人素质差"，就完全跑偏了——它更可能是渠道与预订习惯的差异。

In [ ]:
# ── 出图 D02_03：客源国取消率（带 95% 区间）──
# 为什么加误差线：英国 10,360 条与巴西 1,981 条的区间宽度不同，
# 不画区间会把"样本少所以波动大"读成"这个国家真的更高"。
# 规范落在哪几行：按取消率排序（不是按规模）/ 区间跨组重叠时不作强弱判断 / 横轴百分比
# ---- 1 指标 ----
rank = ct.iloc[:-1].sort_values("取消率", ascending=False)
hi_name, lo_name = rank.index[0], rank.index[-1]
gap = (rank["取消率"].iloc[0] - rank["取消率"].iloc[-1]) * 100
title = f"前十国取消率从 {rank['取消率'].iloc[-1]:.0%} 到 {rank['取消率'].iloc[0]:.0%}，差 {gap:.0f} 个百分点"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"仅前 {a['top_country_n']} 大客源国；误差线为 95% Wilson 区间；"
            f"排序依据是取消率，不是规模")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.4, 4.6))
viz.hbar(ax, rank.index, rank["取消率"].to_numpy(), value_fmt="{:.1%}",
         errors=(rank["下限"].to_numpy(), rank["上限"].to_numpy()), colors=viz.MAIN_MID)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlim(0, rank["上限"].max() * 1.22)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D02_03_客源国取消率.png")
# ---- 4 读回断言：条长等于取消率，且最高组的区间不至于和最低组重叠 ----
bars = ax.patches
assert np.allclose([b.get_width() for b in bars], rank["取消率"].to_numpy())
assert rank["下限"].iloc[0] > rank["上限"].iloc[-1], "首尾两组的区间重叠了，'差多少'要改写"
plt.show()

葡萄牙（37.0%）、巴西（36.7%）、意大利（35.2%）在高端；英国（19.1%）、德国（19.6%）、
荷兰（18.4%）在低端。首尾两组区间不重叠，所以这个差距是真的，不是抽样波动。

但**差距的来源不是国籍**。同样一个国家内部，OTA 客与直订客的取消率能差 20 个百分点，
而客源国与渠道高度相关（某些国家的客源几乎全走 OTA）。这条线在 D06 会正式拆开。

## 4 渠道：谁把客人带来的，比客人来自哪里更能解释取消

"从哪来"的第三层是渠道。这一层比地理更"可动"——但也不是短期能动的，理由见 D06。

In [ ]:
# 渠道结构表：三个维度各看一次规模与取消率
# 口径统一走 rate.rate_table，不在这里另写 SQL 聚合——否则分组规则会和别处对不上
from hotel_insight import rate

for col in ["market_segment", "distribution_channel", "customer_type"]:
    t = rate.rate_table(d, col)
    print(f"\n== {col} ==")
    print(t[["记录数", "占比", "取消率"]].to_string(float_format=lambda v: f"{v:.3f}"))

In [ ]:
# ── 出图 D02_04：市场细分规模与取消率 ──
# 为什么只画 market_segment：distribution_channel 与它高度重合（旅行社渠道≈OTA+线下），
# 画两张会把同一件事说两遍；customer_type 的信息量在取消分析里更弱，放表里就够。
# 规范落在哪几行：剔除样本不足的取值（航空相关 222 条、免费赠房 62 条、未定义 2 条）/
#               同一维度用同一主色 / 取消率带区间
# ---- 1 指标：只保留样本 ≥ 1000 的细分，其余在副标题里交代 ----
from hotel_insight import rate

seg = rate.rate_table(d, "market_segment")
keep = seg[seg["记录数"] >= 1000].sort_values("取消率", ascending=False)
drop = seg[seg["记录数"] < 1000]
ota = seg.loc[a["focus_market"]]
title = f"{a['focus_market']}占 {ota['占比']:.0%}、取消率 {ota['取消率']:.0%}，是取消的主战场"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"仅画样本 ≥1,000 条的分组；另 {len(drop)} 组合计 {int(drop['记录数'].sum()):,} 条未画；"
            f"误差线为 95% Wilson 区间")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.4, 3.6))
viz.hbar(ax, keep.index, keep["取消率"].to_numpy(), value_fmt="{:.1%}",
         errors=(keep["下限"].to_numpy(), keep["上限"].to_numpy()),
         colors=[viz.MAIN if x == a["focus_market"] else viz.MAIN_MID for x in keep.index])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlim(0, keep["上限"].max() * 1.25)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D02_04_市场细分取消率.png")
# ---- 4 读回断言 ----
assert np.allclose([b.get_width() for b in ax.patches], keep["取消率"].to_numpy())
assert keep["下限"].iloc[0] > keep["上限"].iloc[-1], "最高与最低细分的区间重叠了"
plt.show()

三个结论：

1. **OTA 一家占 60%、取消率 35.6%**，是取消问题的主战场。它同时是"规模最大"和"取消率最高"，
   所以超额取消单最多（D05 会算）。
2. **线下旅行社只占 16%，取消率 14.9%**，接近直订。同样是"旅行社"，线上线下的行为差一倍。
   这说明问题不在"通过第三方订"，而在"OTA 的免费取消政策"。
3. **企业协议 12.2%，全场最低**。协议客户有公司约束，退订成本高。

这三条是 D05、D06 的入口：真正能动的不是"客源国"，而是"渠道政策"。

## 5 本册结论

| 问题 | 答案 | 证据 |
| --- | --- | --- |
| 从哪来（地理） | 高度集中：葡萄牙 30.5%，前十国 81.5%，176 国里后 166 国只占 18.5% | 帕累托 |
| 从哪来（渠道） | 旅行社渠道 79.8%，其中 OTA 59.9%；直订 13.5%；企业协议 4.8% | 渠道表 |
| 从哪来（客群） | 散客 82.5%，散客团体 13.3% | 客群表 |
| 与酒店类型的关系 | 爱尔兰/英国/葡萄牙偏度假店；意大利/法国/德国偏城市店 | 堆叠条 |
| 与取消的关系 | 客源国间差 19pp，但来源是渠道与提前期，不是国籍 | 下一册起拆 |

**答不了的**：客人从哪个城市来、酒店开在哪个城市、两个城市之间怎么流动——数据里没有这些字段。

## 6 发现清单（本册）

| # | 发现 | 影响 |
| --- | --- | --- |
| 1 | 客源国缺失 436 条（0.51%） | 分组占比合计不到 1，做结构分解时必须把缺失当成一组（D06 踩过） |
| 2 | 前 8 国里爱尔兰/英国的度假店占比超过一半 | 两条业务线的客源结构不同，合看会互相掩盖 |
| 3 | 客源国取消率与"提前期"高度纠缠 | 不能把国家差异直接当因果，D05 用渠道与提前期交叉验证 |
| 4 | `market_segment` 有 2 条"未定义" | 样本极小，不参与任何比率比较，只在表里保留 |

In [ ]:
con.close()
print("D02 完成：客源三层（地理/渠道/客群）已量化，取消差异留给 D05")